# Module 4 · Text Preprocessing & Tokenization

**Track:** Natural Language Processing (0 to Masterclass)  
**Prerequisites:** Python fundamentals, Basic Probability  
**Difficulty:** ⭐ Beginner → Intermediate  
**Estimated Time:** 90–120 minutes

---



## 🎓 Socratic Deep Check

> *"GPT-4 has a vocabulary of ~100,256 tokens. Why is this number so specific? If we used raw characters, our sequences would be 5x longer. If we used whole words, our vocabulary would be 1,000,000+ words. How do we find the 'Goldilocks' zone between computational efficiency and linguistic expressive power?"*

---

## Why This Matters for AI Engineers

Every LLM interaction starts with tokenization. **API costs are per-token.** Context windows are measured in tokens. RAG chunking (NB21) depends on token boundaries. Even 'traditional' AI tasks like sentiment analysis or classification often fail not because the model is weak, but because the preprocessing was noisy or inconsistent.


---

## 📑 Table of Contents

* [🎓 Socratic Deep Check](#socratic-deep-check)
* [Why This Matters for AI Engineers](#why-this-matters-for-ai-engineers)
* [0 · Setup & Prerequisites](#0-setup-prerequisites)
* [1 · Traditional Preprocessing (The Pre-LLM Era)](#1-traditional-preprocessing-the-pre-llm-era)
  * [1.1 · The Production-Grade Cleaning Pipeline](#11-the-production-grade-cleaning-pipeline)
  * [1.2 · Stemming vs. Lemmatization](#12-stemming-vs-lemmatization)
* [2 · NLP Data Augmentation (EDA)](#2-nlp-data-augmentation-eda)
  * [🎓 Socratic Deep Check](#socratic-deep-check)
  * [2.1 · Why Text Augmentation is Harder](#21-why-text-augmentation-is-harder)
  * [2.2 · Easy Data Augmentation (EDA) Techniques](#22-easy-data-augmentation-eda-techniques)
  * [2.3 · Advanced: Back-Translation](#23-advanced-back-translation)
* [3 · Classical Statistical NLP (The Vector Space Era)](#3-classical-statistical-nlp-the-vector-space-era)
  * [3.1 · Bag-of-Words (BoW)](#31-bag-of-words-bow)
  * [3.2 · TF-IDF: The Math](#32-tf-idf-the-math)
  * [🎓 Concept Bridge: The Sparsity Problem](#concept-bridge-the-sparsity-problem)
* [3.3 · The OOV Problem and the Whitespace Assumption](#33-the-oov-problem-and-the-whitespace-assumption)
  * [1. The OOV (Out-of-Vocabulary) Problem](#1-the-oov-out-of-vocabulary-problem)
  * [2. The Whitespace Assumption](#2-the-whitespace-assumption)
  * [3. The SentencePiece Revolution: Space as a Character](#3-the-sentencepiece-revolution-space-as-a-character)
* [4 · Modern Subword Tokenization](#4-modern-subword-tokenization)
  * [4.1 · Byte-Pair Encoding (BPE)](#41-byte-pair-encoding-bpe)
  * [4.2 · WordPiece vs. Unigram (SentencePiece)](#42-wordpiece-vs-unigram-sentencepiece)
    * [WordPiece (BERT, DistilBERT)](#wordpiece-bert-distilbert)
    * [Unigram / SentencePiece (T5, ALBERT, Llama 3)](#unigram-sentencepiece-t5-albert-llama-3)
  * [🎓 Masterclass Trick: Byte-Fallback](#masterclass-trick-byte-fallback)
  * [4.3 · Subword Regularization for Robustness](#43-subword-regularization-for-robustness)
  * [🎓 Socratic Deep Check](#socratic-deep-check)
    * [The Core Idea: Tokenization as Data Augmentation](#the-core-idea-tokenization-as-data-augmentation)
    * [4.3.1 · BPE Dropout (Provilkov et al., 2020)](#431-bpe-dropout-provilkov-et-al-2020)
    * [4.3.2 · Unigram Sampling (Kudo, 2018)](#432-unigram-sampling-kudo-2018)
    * [4.3.3 · Practical Implementation](#433-practical-implementation)
* [4.4 · Domain Adaptation: Custom Pre-tokenization and Added Tokens](#44-domain-adaptation-custom-pre-tokenization-and-added-tokens)
  * [🎓 Socratic Deep Check](#socratic-deep-check)
  * [The Solution: Isolation & Freezing](#the-solution-isolation-freezing)
* [5 · Production Hands-on: Building a Tokenizer](#5-production-hands-on-building-a-tokenizer)
* [6 · Advanced Tokenization: Byte-Level BPE & Tokenizer Bias](#6-advanced-tokenization-byte-level-bpe-tokenizer-bias)
  * [6.1 · Byte-Level BPE (BBPE)](#61-byte-level-bpe-bbpe)
  * [6.2 · Tokenizer Parity and Language Bias](#62-tokenizer-parity-and-language-bias)
  * [🎓 Socratic Deep Check](#socratic-deep-check)
  * [6.3 · Production Consequences of Tokenizer Bias](#63-production-consequences-of-tokenizer-bias)
* [7 · Unicode Normalization: The Silent Danger of Raw Text](#7-unicode-normalization-the-silent-danger-of-raw-text)
  * [🎓 Socratic Deep Check](#socratic-deep-check)
  * [7.1 · The Unicode Pitfall](#71-the-unicode-pitfall)
  * [🎓 Masterclass Insight: Why Raw Text is Dangerous](#masterclass-insight-why-raw-text-is-dangerous)
* [8 · The Tokenizer-Free Paradigm: CANINE & ByT5](#8-the-tokenizer-free-paradigm-canine-byt5)
  * [🎓 Socratic Deep Check](#socratic-deep-check)
  * [8.1 · Breaking the BPE Bottleneck](#81-breaking-the-bpe-bottleneck)
  * [8.2 · Why Bytes are Theoretically Superior](#82-why-bytes-are-theoretically-superior)
* [✅ Knowledge Check](#knowledge-check)
  * [Q1: The OOV Paradox](#q1-the-oov-paradox)
  * [Q2: Mathematical Efficiency](#q2-mathematical-efficiency)
  * [Q3: Byte-Fallback (Senior Level)](#q3-byte-fallback-senior-level)
* [🔨 Practical Challenge](#practical-challenge)


## 0 · Setup & Prerequisites

In [ ]:
# 1. Install necessary libraries
!pip install -q nltk spacy scikit-learn transformers tokenizers tiktoken

# 2. Download NLTK data and spaCy model
import nltk
import spacy
import subprocess

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

try:
    nlp = spacy.load('en_core_web_sm')
except OSError:
    subprocess.run(['python', '-m', 'spacy', 'download', 'en_core_web_sm'])
    nlp = spacy.load('en_core_web_sm')

print("✅ Setup Complete: Dependencies installed and models downloaded.")

---
## 1 · Traditional Preprocessing (The Pre-LLM Era)

Before we look at learned tokenization, we must handle the raw noise in human text. While modern LLMs are robust, traditional ML and search engines (ElasticSearch/OpenSearch) still rely heavily on these deterministic filters.

### 1.1 · The Production-Grade Cleaning Pipeline
We use **Unicode Normalization** to handle different encodings of the same character (e.g., 'e' + '´' vs 'é').

In [ ]:
import re
import unicodedata
from typing import Optional

def production_clean(text: str, aggressive: bool = False) -> str:
    """
    A robust cleaning function for production pipelines.
    
    Args:
        text (str): The raw input string.
        aggressive (bool): If True, removes all non-alphanumeric characters. Defaults to False.
        
    Returns:
        str: The normalized and cleaned text.
    """
    # 1. Unicode Normalization (NFKD separates base chars from accents)
    text = unicodedata.normalize('NFKD', text)
    
    # 2. Lowercasing (Standardize casing)
    text = text.lower()
    
    # 3. Strip HTML Tags (Regex-based)
    text = re.sub(r'<[^>]*>', ' ', text)
    
    # 4. Handle Emojis & Weird Punctuation (Optional)
    if aggressive:
        # Remove everything except alphanumeric and spaces
        text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    
    # 5. Collapse Whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# TEST IT
raw_input = "  <p>The résumé was <b>PERFECT</b>! 🎉  \n Contact: me@example.com </p> "
print(f"Raw: {raw_input}")
print(f"Cleaned (Balanced):  {production_clean(raw_input)}")
print(f"Cleaned (Aggressive): {production_clean(raw_input, aggressive=True)}")

### 1.2 · Stemming vs. Lemmatization

| Feature | Stemming (e.g., Porter) | Lemmatization (e.g., spaCy) |
|---------|-------------------------|---------------------------|
| **Logic** | Hops off suffixes (Heuristic) | Uses dictionary & context (Morphological) |
| **Speed** | 🚀 Extremely Fast | 🐢 Slower (requires POS tagging) |
| **Accuracy**| Low ("studies" → "studi") | High ("studies" → "study") |
| **Usecase** | High-speed indexing, search | Complex NLP, Chatbots |

In [ ]:
from nltk.stem import PorterStemmer
import spacy
from typing import List

def compare_reduction(words: List[str]) -> None:
    """
    Compares Stemming and Lemmatization for a list of words.
    """
    stemmer = PorterStemmer()
    nlp = spacy.load("en_core_web_sm")

    print(f"{ 'Word':<12} | { 'Stemming':<12} | { 'Lemmatization':<12}")
    print("-" * 45)
    for w in words:
        stem = stemmer.stem(w)
        lemma = nlp(w)[0].lemma_
        print(f"{w:<12} | {stem:<12} | {lemma:<12}")

test_words = ["changing", "changes", "changed", "charger", "charge"]
compare_reduction(test_words)

---
## 2 · NLP Data Augmentation (EDA)

In Computer Vision, augmentation is straightforward: rotate a cat, and it's still a cat. In NLP, flipping a word or changing a character can shift the entire sentiment or meaning. 

### 🎓 Socratic Deep Check
> *"If I replace 'good' with 'bad' in a movie review, the grammar is perfect, but the label is now wrong. If I replace 'good' with 'great', the label is preserved. How do we automate this without a human in the loop?"*

### 2.1 · Why Text Augmentation is Harder
1. **Discreteness:** Images are continuous pixels; text is discrete tokens. There is no "middle ground" between two words.
2. **Semantic Sensitivity:** Small changes (e.g., adding "not") = big meaning shifts.
3. **Syntactic Fragility:** Swapping words blindly can make a sentence ungrammatical.

### 2.2 · Easy Data Augmentation (EDA) Techniques
Wei and Zou (2019) proposed four simple operations to boost performance on small datasets: **Synonym Replacement**, **Random Insertion**, **Random Swapping**, and **Random Deletion**.

In [ ]:
import random
from typing import List, Set
from nltk.corpus import wordnet

class TextAugmentor:
    """
    A production-grade implementation of Easy Data Augmentation (EDA) techniques.
    """
    
    def get_synonyms(self, word: str) -> List[str]:
        """
        Find synonyms using WordNet.
        
        Args:
            word (str): The word to find synonyms for.
        Returns:
            List[str]: A list of unique synonyms.
        """
        synonyms = set()
        for syn in wordnet.synsets(word):
            for l in syn.lemmas():
                synonym = l.name().replace('_', ' ').replace('-', ' ').lower()
                if synonym != word.lower():
                    synonyms.add(synonym)
        return list(synonyms)

    def synonym_replacement(self, sentence: str, n: int = 1) -> str:
        """
        Replaces n random non-stop words with their synonyms.
        """
        words = sentence.split()
        new_words = words.copy()
        random_word_list = list(set([word for word in words if word.isalnum()]))
        random.shuffle(random_word_list)
        
        num_replaced = 0
        for random_word in random_word_list:
            synonyms = self.get_synonyms(random_word)
            if len(synonyms) >= 1:
                synonym = random.choice(synonyms)
                new_words = [synonym if word == random_word else word for word in new_words]
                num_replaced += 1
            if num_replaced >= n:
                break

        return ' '.join(new_words)

    def random_deletion(self, sentence: str, p: float = 0.2) -> str:
        """
        Randomly deletes words with probability p.
        """
        words = sentence.split()
        if len(words) == 1: return sentence
        
        new_words = [w for w in words if random.uniform(0, 1) > p]
        
        # Ensure at least one word remains
        if len(new_words) == 0:
            return random.choice(words)
        return ' '.join(new_words)

augmentor = TextAugmentor()
sample = "The product was incredibly easy to use and fast."

print(f"Original: {sample}")
print(f"Synonym Replacement: {augmentor.synonym_replacement(sample, n=2)}")
print(f"Random Deletion:      {augmentor.random_deletion(sample, p=0.3)}")

### 2.3 · Advanced: Back-Translation

The gold standard for semantic-preserving augmentation. We translate a sentence to a pivot language (e.g., German) and then back to the original language. This often results in a sentence with the same meaning but different phrasing.

> **Why Back-Translation?** It captures syntactic variety and paraphrasing that simple word swapping cannot.

In [ ]:
from transformers import MarianMTModel, MarianTokenizer
from typing import Tuple
import torch

def get_marian_bt_models(src_lang: str = "en", pivot_lang: str = "de") -> Tuple[MarianTokenizer, MarianMTModel, MarianTokenizer, MarianMTModel]:
    """
    Utility to load source -> pivot and pivot -> source translation models.
    """
    fwd_model_name = f"Helsinki-NLP/opus-mt-{src_lang}-{pivot_lang}"
    rev_model_name = f"Helsinki-NLP/opus-mt-{pivot_lang}-{src_lang}"
    
    fwd_tokenizer = MarianTokenizer.from_pretrained(fwd_model_name)
    fwd_model = MarianMTModel.from_pretrained(fwd_model_name)
    
    rev_tokenizer = MarianTokenizer.from_pretrained(rev_model_name)
    rev_model = MarianMTModel.from_pretrained(rev_model_name)
    
    return fwd_tokenizer, fwd_model, rev_tokenizer, rev_model

def back_translate(text: str, fwd_tok, fwd_mod, rev_tok, rev_mod) -> str:
    """
    Performs EN -> DE -> EN back-translation.
    """
    # 1. English to German
    inputs = fwd_tok(text, return_tensors="pt", padding=True)
    with torch.no_grad():
        translated = fwd_mod.generate(**inputs)
    german_text = fwd_tok.decode(translated[0], skip_special_tokens=True)
    
    # 2. German back to English
    inputs = rev_tok(german_text, return_tensors="pt", padding=True)
    with torch.no_grad():
        back_translated = rev_mod.generate(**inputs)
    return rev_tok.decode(back_translated[0], skip_special_tokens=True)

print("⏳ Loading Back-Translation models (Helsinki-NLP)...")
# Note: These models represent ~600MB of downloads.
# fwd_t, fwd_m, rev_t, rev_m = get_marian_bt_models()
# bt_sample = "The quick brown fox jumps over the lazy dog."
# print(f"Original: {bt_sample}")
# print(f"Back-Translated: {back_translate(bt_sample, fwd_t, fwd_m, rev_t, rev_m)}")

---
## 3 · Classical Statistical NLP (The Vector Space Era)

How do we turn words into numbers without deep learning? For decades, **TF-IDF** was the gold standard for information retrieval.

### 3.1 · Bag-of-Words (BoW)
A simple count of word frequencies. 
**Problem:** It treats "the" (very common, low info) as more important than "quantum" (rare, high info).

### 3.2 · TF-IDF: The Math

$$TF(t, d) = \frac{\text{Count of term } t \text{ in doc } d}{\text{Total words in doc } d}$$

$$IDF(t, D) = \log \left( \frac{\text{Total docs in } D}{1 + \text{Docs containing } t} \right)$$

$$TF\text{-}IDF = TF \times IDF$$

Why the log? It dampens the effect of frequency, ensuring that having a term 100 times isn't 100x more important than having it once.

In [ ]:
import math
from collections import Counter
import pandas as pd
from typing import Dict, List
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = [
    "the cat sat on the mat",
    "the dog sat on the rug",
    "the cat played with the dog"
]

def calculate_tfidf_scratch(documents: List[str]) -> Dict[str, float]:
    """
    Calculates TF-IDF for the first document in a corpus for demonstration.
    """
    # 1. Collect all terms
    all_words = set(" ".join(documents).split())
    N = len(documents)
    
    # 2. IDF calculation
    idf = {}
    for word in all_words:
        doc_count = sum(1 for d in documents if word in d.split())
        idf[word] = math.log(N / (doc_count))
    
    # 3. TF-IDF Calculation for the first document
    doc0 = documents[0].split()
    counts = Counter(doc0)
    results = {}
    for word in set(doc0):
        tf = counts[word] / len(doc0)
        results[word] = tf * idf[word]
    return results

print("TF-IDF (Scratch) for Doc 0:")
print(calculate_tfidf_scratch(corpus))

# The Scikit-Learn Way
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)
print("\nSklearn Vocabulary Size:", len(vectorizer.get_feature_names_out()))

### 🎓 Concept Bridge: The Sparsity Problem

**The Senior Perspective:** Imagine a vocabulary of 50,000 words. Every sentence in your dataset is now represented as a vector of length 50,000. 
*   99.9% of that vector will be zeros. 
*   This is **The Curse of Dimensionality**. 
*   Models struggle to learn patterns when the input space is so sparse.

**This is precisely why we transitioned to Embeddings (Dense Vectors) and Subword Tokenization.**

---
## 3.3 · The OOV Problem and the Whitespace Assumption

Before we dive into modern algorithms, we must understand the "cracks" in classical systems that forced the invention of subword tokenization.

### 1. The OOV (Out-of-Vocabulary) Problem
In the era of Word2Vec and GloVe, we built a static dictionary of the top $V$ words (e.g., 50,000). Any word not in this list was mapped to a single, catch-all token: `<UNK>`.

**Why this failed:**
- **Morphologic Richness:** In languages like Turkish, Finnish, or German, a single root can have thousands of inflected forms (e.g., *dog*, *dogs*, *dog's*, *doggedly*). A static dictionary would either need to be infinite or lose massive amounts of information to the `<UNK>` bucket.
- **The "Great Wall" of UNKs:** If a model sees `<UNK> <UNK> sat on the <UNK>`, it has zero semantic signal to work with.

### 2. The Whitespace Assumption
Classical tokenizers (like NLTK's `word_tokenize`) rely on a fundamental western-centric assumption: **words are separated by spaces.**

**Why this is a "False Idol":**
- **Logographic Languages:** Chinese and Japanese do not use spaces. `我喜欢学习NLP` (I like studying NLP) has no separators.
- **Thai & Lao:** These languages use "scriptio continua" where characters flow without breaks.
- **German:** Compound words like *Donaudampfschifffahrtselektrizitätenhauptbetriebswerkbauunterbeamtengesellschaft* (yes, that's one word) break whitespace-based logic.

### 3. The SentencePiece Revolution: Space as a Character
How do we build one tokenizer for the entire world? **SentencePiece** (Kudo & Richardson, 2018) changed the game by treating the input as a raw stream of characters.

Instead of stripping whitespace first, SentencePiece treats the **space itself as just another character**, usually represented as the "meta-symbol" `_` (Unicode `U+2581`).

**The Pipeline Unification:**
- **Traditional:** `Hello world` → `["Hello", "world"]` (Space is lost/used as a separator)
- **SentencePiece:** `Hello world` → `["_Hello", "_world"]` (Space is preserved as part of the token)

> **🎓 Masterclass Insight:** By treating space as a character, SentencePiece becomes **completely language-agnostic**. It doesn't care if your language uses spaces, dashes, or nothing at all. It simply finds the most statistically significant subword units across the entire character stream. This is why it is the default choice for truly global models like T5 and Llama.


---
## 4 · Modern Subword Tokenization

Why didn't word-level tokenization survive the LLM era?
1. **Vocabulary Bloom:** Can't store 1M+ words in GPU memory.
2. **OOV (Out-of-Vocabulary):** If the model sees 'Unbelievable' but only trained on 'Believable', it's stuck.

**Solution:** Break words into meaningful sub-units.

### 4.1 · Byte-Pair Encoding (BPE)
*Used by: GPT-2, GPT-3, GPT-4, Llama 2/3.*

**Logic:**
1. Split words into individual bytes/characters.
2. Count the most frequent adjacent pair (e.g., 't' + 'h').
3. Merge them into a single token ('th').
4. Repeat until you reach your target `vocab_size`.

In [ ]:
from collections import Counter
import re
from typing import Dict, Tuple

def get_stats(vocab: Dict[str, int]) -> Counter:
    """Counts frequency of adjacent symbol pairs."""
    pairs = Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols)-1):
            pairs[symbols[i], symbols[i+1]] += freq
    return pairs

def merge_vocab(pair: Tuple[str, str], v_in: Dict[str, int]) -> Dict[str, int]:
    """Merges a symbol pair into a single symbol in the vocabulary."""
    v_out = {}
    bigram = re.escape(' '.join(pair))
    p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    for word in v_in:
        w_out = p.sub(''.join(pair), word)
        v_out[w_out] = v_in[word]
    return v_out

# Initial Vocab (words split into chars space-separated)
vocab = {'l o w </w>' : 5, 'l o w e r </w>' : 2, 'n e w e s t </w>': 6, 'w i d e s t </w>': 3}
num_merges = 5

for i in range(num_merges):
    pairs = get_stats(vocab)
    if not pairs: break
    best = max(pairs, key=pairs.get)
    vocab = merge_vocab(best, vocab)
    print(f"Merge {i+1}: {best} -> freq={pairs[best]}")
    print(f"Current state: {vocab}\n")

### 4.2 · WordPiece vs. Unigram (SentencePiece)

#### WordPiece (BERT, DistilBERT)
Similar to BPE but with a twist: instead of most frequent pair, it picks the pair that **maximizes likelihood**. 
$$Score = \frac{P(AB)}{P(A)P(B)}$$

#### Unigram / SentencePiece (T5, ALBERT, Llama 3)
It works in reverse! It starts with a HUGE vocabulary and **removes** pieces that don't contribute much to the probability of the corpus.

### 🎓 Masterclass Trick: Byte-Fallback

Older tokenizers (like BERT's) used an `[UNK]` token for any character not in their vocabulary (e.g., rare emojis like 🫠 or non-Latin scripts). 

**SentencePiece** (used by Llama models) solved this with **Byte-Fallback**:
1. The vocabulary pre-includes the **256 individual UTF-8 bytes**.
2. If a character is Out-of-Vocabulary, the tokenizer falls back to encoding it as its raw byte sequence.
3. **Impact:** No more `[UNK]` tokens. Every possible character string can be encoded perfectly, which is critical for multilingual and code-heavy datasets.

### 4.3 · Subword Regularization for Robustness

### 🎓 Socratic Deep Check
> *"We have spent this entire chapter treating tokenization as a deterministic function: one input, one output. But what if the 'correct' tokenization doesn't exist? The word 'unbelievable' could be tokenized as `['un', 'believ', 'able']`, or `['un', 'believe', 'able']`, or `['unbeliev', 'able']`. Each is a valid decomposition. If we always pick the same one, are we leaving information on the table?"*

---

#### The Core Idea: Tokenization as Data Augmentation

Standard BPE and WordPiece are **deterministic**: given a word, they always produce the exact same subword sequence. This means the model only ever sees one possible segmentation of each word, and it can become **brittle** — overfitting to that specific decomposition.

**Subword Regularization** (Kudo, 2018) reframes tokenization as a **stochastic process**. Instead of committing to a single 'best' segmentation, we **sample from multiple valid segmentations** during training. This acts as a powerful form of **data augmentation at the tokenization layer itself**, forcing the model to learn meaning from *multiple representations* of the same word.

**Why does this work?**
- The model can no longer memorize a fixed token-to-meaning mapping.
- It must learn **compositional understanding** — inferring meaning from varying subword contexts.
- This is analogous to how **dropout** prevents co-adaptation between neurons: subword regularization prevents co-adaptation between token positions.

There are two major implementations of this idea:

---

#### 4.3.1 · BPE Dropout (Provilkov et al., 2020)

Standard BPE applies merge operations greedily in a fixed order. **BPE Dropout** introduces a simple but devastating perturbation: at each merge step during tokenization, **randomly skip (drop) the merge with probability `p`**.

| Step | Standard BPE | BPE Dropout (`p=0.1`) |
|------|-------------|----------------------|
| Input | `u n b e l i e v a b l e` | `u n b e l i e v a b l e` |
| Merge 1 | `u n b e l ie v a b l e` | `u n b e l ie v a b l e` |
| Merge 2 | `u n b e liev a b le` | ⚡ *Skip! (dropped)* |
| Merge 3 | `un b eliev ab le` | `u n b e liev a b le` |
| ... | `un believ able` | `un b eliev ab le` |
| **Result** | `['un', 'believ', 'able']` | `['un', 'b', 'eliev', 'ab', 'le']` |

**Key Properties:**
- **`p = 0`** → Deterministic (standard BPE). This is used at **inference** time.
- **`p = 0.1`** → Recommended default for training. Provides a good balance between diversity and coherence.
- **`p = 1.0`** → Every merge is skipped, producing a pure character-level tokenization.
- The result is that the **same sentence produces different token sequences on every training epoch**, providing free data augmentation with zero computational overhead.

---

#### 4.3.2 · Unigram Sampling (Kudo, 2018)

The Unigram model (used by SentencePiece in T5, ALBERT, and Llama 3) is *inherently probabilistic*. It maintains a probability for every subword piece in the vocabulary, and the probability of a segmentation $\mathbf{x} = (x_1, x_2, \ldots, x_k)$ is:

$$P(\mathbf{x}) = \prod_{i=1}^{k} P(x_i)$$

The **Viterbi** (most probable) segmentation is what you get during deterministic inference. But during training, we can **sample** from the $n$-best segmentations according to their probabilities.

**Example:** For the word `'unbelievable'`, the Unigram model might have:

| Segmentation | Log-Probability |
|---|---|
| `['▁un', 'believ', 'able']` | -8.23 (Most Probable → Viterbi) |
| `['▁un', 'believe', 'able']` | -8.91 |
| `['▁unbeliev', 'able']` | -9.14 |
| `['▁un', 'bel', 'iev', 'able']` | -10.67 |

With **`nbest_size > 1`** and a **temperature** parameter `alpha`:
- **`alpha = 0`** → Uniform sampling from the top-k (maximum diversity).
- **`alpha = 1.0`** → Sample proportional to the true probabilities (calibrated).
- **`alpha → ∞`** → Always pick the Viterbi path (deterministic).

> **🎓 The Masterclass Insight:** BPE Dropout and Unigram Sampling achieve the same goal — **injecting noise at the tokenization layer** — but from opposite directions. BPE Dropout *removes* structure (skipping merges). Unigram Sampling *adds* stochasticity (sampling from a probability distribution). Both are forms of **regularization**, and both produce measurable improvements on low-resource translation and noisy text classification tasks.

---

#### 4.3.3 · Practical Implementation

In [ ]:
# =============================================================================
# 4.3.3 · Subword Regularization in Practice
# =============================================================================
# We demonstrate both approaches using industry-standard libraries.

# --- Method 1: BPE Dropout with HuggingFace `tokenizers` ----------------------
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
import json
import tempfile
import os

# Train a small BPE tokenizer for demonstration
training_corpus = [
    "The unbelievable power of subword tokenization is transforming NLP.",
    "Regularization prevents overfitting and improves generalization.",
    "Dropout is a simple but powerful technique for neural network training.",
    "Subword algorithms break words into meaningful units for better coverage.",
    "The unbelievable advancements in natural language processing continue.",
    "Natural language processing relies on tokenization and subword models.",
    "Better tokenization leads to better language understanding and generation.",
]

bpe_tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
bpe_tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(special_tokens=["[UNK]", "[PAD]"], vocab_size=200)
bpe_tokenizer.train_from_iterator(training_corpus, trainer)

# Save → inject dropout into config → reload
# This is the production pattern for enabling BPE Dropout.
with tempfile.TemporaryDirectory() as tmpdir:
    path = os.path.join(tmpdir, 'tokenizer.json')
    bpe_tokenizer.save(path)
    
    # Inject dropout=0.1 into the saved BPE model config
    with open(path, 'r') as f:
        config = json.load(f)
    config['model']['dropout'] = 0.1  # ← The key parameter
    with open(path, 'w') as f:
        json.dump(config, f)
    
    # Reload with dropout active
    bpe_dropout_tokenizer = Tokenizer.from_file(path)

test_word = "unbelievable"
print("=" * 60)
print("BPE DROPOUT DEMONSTRATION")
print("=" * 60)
print(f"Input: '{test_word}'\n")
print("Multiple tokenizations of the SAME word (dropout=0.1):")
for i in range(5):
    tokens = bpe_dropout_tokenizer.encode(test_word).tokens
    print(f"  Run {i+1}: {tokens}")

print("\n→ Notice: each run may produce a DIFFERENT segmentation.")
print("  At inference time, set dropout=0.0 for deterministic output.")

# --- Method 2: Unigram Sampling with SentencePiece ---------------------------
# SentencePiece natively supports nbest_size and alpha for Unigram sampling.
# Below is the API pattern you would use with a trained .model file.

print("\n" + "=" * 60)
print("UNIGRAM SAMPLING (SentencePiece API Pattern)")
print("=" * 60)

sentencepiece_example = '''
import sentencepiece as spm

# Load a pre-trained SentencePiece model (e.g., from Llama, T5)
sp = spm.SentencePieceProcessor(model_file='my_model.model')

# --- Deterministic (Inference) ---
# Viterbi decoding: always returns the single most probable segmentation
tokens = sp.encode('unbelievable', out_type=str)
# → ['▁un', 'believ', 'able']

# --- Stochastic (Training with Subword Regularization) ---
# nbest_size: sample from top-k segmentations (use -1 for all)
# alpha:       temperature (0.0 = uniform, 1.0 = proportional to prob)
tokens = sp.encode(
    'unbelievable',
    out_type=str,
    enable_sampling=True,   # ← Enable subword regularization
    nbest_size=-1,          # ← Sample from ALL valid segmentations
    alpha=0.1               # ← Smoothing temperature
)
# → Different result each call: ['▁un', 'believe', 'able']
#                            or: ['▁unbeliev', 'able']
#                            or: ['▁un', 'bel', 'iev', 'able']
'''
print(sentencepiece_example)
print("💡 Key Takeaway:")
print("   During TRAINING  → enable_sampling=True  (regularization ON)")
print("   During INFERENCE → enable_sampling=False (deterministic, reproducible)")

---
## 4.4 · Domain Adaptation: Custom Pre-tokenization and Added Tokens

### 🎓 Socratic Deep Check
> *"Imagine you are processing medical records. The string `[PATIENT_ID]` appears frequently as a critical structural marker. If you use a standard BPE tokenizer, it will likely shred this into `['[', 'PAT', 'IENT', '_', 'ID', ']']`. How does this impact the model's ability to recognize the 'atomicity' of this identifier? If the model sees individual brackets and sub-words, can it reliably preserve the structure at scale?"*

In production, generic tokenizers often fail on **domain-specific structural units** like HTML tags, LaTeX formulas, or proprietary identifiers. When BPE 'shreds' these units, we lose semantic coherence and increase the risk of the model 'hallucinating' within the identifier boundaries.

### The Solution: Isolation & Freezing

To solve this, we use two advanced techniques in the HuggingFace `tokenizers` ecosystem:

1.  **Custom Regex Pre-tokenization**: We use a `Split` pre-tokenizer to find these patterns *before* the BPE algorithm sees them. By setting the behavior to `behavior="isolated"`, we tell the tokenizer: "Every time you see this regex match, stop and make it its own unbreakable token."
2.  **Added Tokens (`AddedToken`)**: We explicitly add these strings to the vocabulary. By marking them as `special=True`, we ensure they are treated as atomic units that the BPE algorithm is forbidden from splitting into smaller subwords, regardless of its frequency statistics.

> **Masterclass Insight:** Adding tokens *without* custom pre-tokenization can sometimes fail if the BPE trainer finds extremely frequent sub-parts. Using the `Split` pre-tokenizer is the 'heavy hammer' that guarantees structural isolation.

In [ ]:
# =============================================================================
# 4.4.1 · Implementing Custom Pre-tokenization for Structural Text
# =============================================================================
from tokenizers import Tokenizer, AddedToken
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Split, Whitespace
import regex as re

# 1. The Scenario: Processing text with critical HTML and ID markers
raw_text = "Document: <div class='med'>[PATIENT_ID_402]</div> contains sensitive data."

# 2. Advanced Pre-tokenization Strategy
# We define patterns for HTML tags and our custom Patient ID bracket format.
html_pattern = r"<[^>]+>"
id_pattern = r"\[PATIENT_ID_\d+\]"
combined_regex = f"({html_pattern}|{id_pattern})"

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

# We use 'Split' to isolate our matches. 
# 'behavior="isolated"' ensures the match itself becomes a token.
tokenizer.pre_tokenizer = Split(re.compile(combined_regex), behavior="isolated")

# 3. Adding Atomic Tokens
# We freeze these tokens to ensure they are NEVER fragmented into subwords.
special_markers = [
    AddedToken("<div class='med'>", special=True), 
    AddedToken("</div>", special=True)
]
tokenizer.add_tokens(special_markers)

output = tokenizer.encode(raw_text)

print(f"Raw Text: {raw_text}")
print(f"\nTokens Produced:")
for t in output.tokens:
    print(f"  -> '{t}'")

print("\n✅ Structural Integrity: Notice how '[PATIENT_ID_402]' and tags are atomic units.")

---
## 5 · Production Hands-on: Building a Tokenizer

In industry, we use the `tokenizers` library (written in Rust) because performance is critical. Training a BPE tokenizer on a 1GB dataset shouldn't take hours.

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# 1. Initialize an empty BPE model
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

# 2. Trainer Config
trainer = BpeTrainer(special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"], 
                    vocab_size=1000) 

# 3. Training Data
data = [
    "The quick brown fox jumps over the lazy dog.",
    "Tokenization is the foundation of NLP engineering.",
    "Modern subword algorithms handle out-of-vocabulary words intelligently."
]

# 4. Train
tokenizer.train_from_iterator(data, trainer)

# 5. Test it!
test_sentence = "The foundation of NLP is tokenization."
output = tokenizer.encode(test_sentence)
print(f"Sentence: {test_sentence}")
print(f"Tokens:   {output.tokens}")
print(f"IDs:      {output.ids}")

---
## 6 · Advanced Tokenization: Byte-Level BPE & Tokenizer Bias

The transition from standard BPE to **Byte-Level BPE (BBPE)** was a pivotal moment in the history of LLMs. It represents the ultimate solution to the Out-Of-Vocabulary (OOV) problem.

### 6.1 · Byte-Level BPE (BBPE)
While standard BPE operates on Unicode characters, BBPE (introduced by OpenAI for GPT-2) operates on **raw UTF-8 bytes**.

**The Mathematical Elegance:**
- A standard BPE tokenizer might have a base vocabulary of thousands of characters to cover multiple languages.
- BBPE realizes that every character is just a sequence of bytes. By including the **256 fundamental bytes** as the base vocabulary, the model can construct *any* string.
- **Result:** The OOV problem is **mathematically eradicated**. If a word is unknown, it is simply decomposed into its constituent bytes.

> **Production Impact:** Models like GPT-3, GPT-4, and RoBERTa use BBPE. This allows them to handle code, emojis, and rare languages with the same architecture, though at the cost of longer sequences for non-Latin scripts.

### 6.2 · Tokenizer Parity and Language Bias

### 🎓 Socratic Deep Check
> *"If an LLM charges $0.01 per 1,000 tokens, and 'Hello' is 1 token in English but 3 tokens in your native language, are you being charged more for the same meaning? Is tokenization inherently biased?"*

This is the hidden tax of modern NLP: **Tokenizer Parity**. Most modern tokenizers are trained predominantly on English-heavy corpora. Consequently, they learn high-level tokens for English words but must rely on character or byte-level decomposition for other languages.

In [ ]:
import tiktoken

def compare_token_efficiency(text_en: str, text_other: str, language: str):
    """
    Demonstrates the tokenization parity between English and other languages.
    Uses 'o200k_base' (GPT-4o) tokenizer.
    """
    enc = tiktoken.get_encoding("o200k_base")
    
    tokens_en = enc.encode(text_en)
    tokens_other = enc.encode(text_other)
    
    print(f"--- Language Comparison: English vs. {language} ---")
    print(f"English: '{text_en}'")
    print(f"Tokens: {len(tokens_en)} -> {tokens_en}")
    
    print(f'\n{language}: "{text_other}"')
    print(f"Tokens: {len(tokens_other)} -> {tokens_other}")
    
    ratio = len(tokens_other) / len(tokens_en)
    print(f"\nRatio: {ratio:.2f}x more tokens for the same meaning.")
    print('-' * 50)

# Example: "The weather is beautiful today."
compare_token_efficiency(
    "The weather is beautiful today.", 
    "الطقس جميل اليوم.", 
    "Arabic"
)

# Example: "I am learning AI engineering."
compare_token_efficiency(
    "I am learning AI engineering.",
    "私はAIエンジニアリングを学んでいます。",
    "Japanese"
)

### 6.3 · Production Consequences of Tokenizer Bias

The disparity in token counts isn't just a linguistic curiosity; it has massive downstream impacts for AI engineers:

1.  **Significantly Higher API Costs:** Since providers charge per token, a user querying in Arabic or Japanese might pay **2x to 5x more** for the exact same information compared to an English speaker.
2.  **Reduced Effective Context Window:** If a model has a 128k context window, an English prompt might use 10% of it, while a Japanese prompt with the same density of information might consume 40%.
3.  **Slower Inference:** LLM generation time is proportional to the number of tokens generated. A response in a "token-expensive" language will take longer to compute and stream.
4.  **Information Density Mismatch:** The model has fewer "layers of abstraction" for languages it tokenizes poorly, potentially leading to lower performance on complex reasoning tasks in those languages.

> **Masterclass Insight:** When building global AI products, choosing a tokenizer with high **multilingual fertility** (like the Llama 3 tokenizer or GPT-4o's o200k_base) is a critical engineering decision that affects your P&L and UX.

---
## 7 · Unicode Normalization: The Silent Danger of Raw Text

### 🎓 Socratic Deep Check
> *"Are 'e' + '´' and 'é' the same? To a human eye, yes. To a standard BPE tokenizer, they might be three tokens vs. one. If your model sees 'résumé' in training as one sequence of bytes and 're\u0301sume\u0301' in inference as another, will your RAG system find the match? What happens when a malicious actor uses 'invisible' characters to bypass your guardrails?"*

### 7.1 · The Unicode Pitfall
Unicode allows multiple ways to represent the same glyph. This flexibility, while necessary for global scripts, is a hidden minefield for deterministic tokenizers.

- **NFD (Decomposition):** Separates base characters from their accents (e.g., `é` → `e` + `◌́`).
- **NFC (Composition):** Combines them back into a single code point (e.g., `e` + `◌́` → `é`).
- **NFKC/NFKD (Compatibility):** Handles stylized variants (e.g., bold or circled numbers) by converting them to their standard equivalents. NFKC is particularly aggressive but often necessary for cleaning user input.

### 🎓 Masterclass Insight: Why Raw Text is Dangerous
1. **Semantic Drift in RAG:** If your Vector Database isn't normalized, a query for "résumé" (NFC) will never match a document containing "re\u0301sume\u0301" (NFD), even though they are visually identical.
2. **Adversarial Injection:** Attackers use **Zero-Width Joiners (ZWJ)** or **Zero-Width Space (ZWSP)** to insert invisible characters between letters. This can bypass keyword-based safety filters (e.g., `B [ZWJ] A [ZWJ] D` looks like `BAD` to a human but `B-ZWJ-A-ZWJ-D` to a regex filter).
3. **Tokenizer Fragmentation:** Non-normalized text forces the tokenizer to use expensive byte-level fallbacks, increasing token counts and reducing the effective context window.


In [ ]:
import unicodedata

def demonstrate_normalization_masterclass():
    # 1. Visual Identity vs. Logical Difference
    # s1 is composed (\u00e9), s2 is decomposed (e + \u0301)
    s1 = "resumé"
    s2 = "resume\u0301"
    
    print(f"--- Visual Identity Check ---")
    print(f"String 1: {s1}")
    print(f"String 2: {s2}")
    print(f"Direct Comparison (s1 == s2): {s1 == s2}")
    
    # 2. The Solution: Normalization
    n1 = unicodedata.normalize('NFC', s1)
    n2 = unicodedata.normalize('NFC', s2)
    print(f"\nAfter NFC Normalization: {n1 == n2}")
    
    # 3. The Compatibility Trap: NFKC
    # circled numbers or fractions
    s_weird = "\u2460 and \u00bd" # ① and ½
    n_k = unicodedata.normalize('NFKC', s_weird)
    print(f"\nNFKC Compatibility: '{s_weird}' becomes '{n_k}'")
    
    # 4. The Zero-Width Danger
    s_poison = "S\u200bT\u200bO\u200bP" # 'STOP' with Zero-Width Spaces
    print(f"\nPoisoned String: '{s_poison}' (Length: {len(s_poison)})")
    print(f"Traditional 'STOP' Match: {'STOP' in s_poison}") # False!

demonstrate_normalization_masterclass()

---
## 8 · The Tokenizer-Free Paradigm: CANINE & ByT5

### 🎓 Socratic Deep Check
> *"If the ultimate goal of tokenization is to provide the model with a sequence of meaningful units, why do we use a fixed vocabulary at all? If a human can read 'Th3 qu1ck br0wn f0x' despite the typos, why can't a model? Is the 'BPE bottleneck' preventing our models from truly understanding Morphology?"*

While Subword Tokenization (BPE/WordPiece) is the current industry standard, it is a lossy heuristic. In the quest for truly universal models, researchers have moved toward **Tokenizer-Free** architectures.

### 8.1 · Breaking the BPE Bottleneck
- **CANINE (Clark et al., 2021):** A model that operates on Unicode scalar values without any fixed vocabulary. It uses a deep transformer to learn its own latent abstractions from raw characters.
- **ByT5 (Lintz et al., 2021):** A version of T5 modified to operate entirely on raw UTF-8 bytes. It doesn't care about 'words' or 'subwords'—only the 256 bytes.

### 8.2 · Why Bytes are Theoretically Superior
1. **Morphological Excellence:** In "agglutinative" languages (like Turkish or Finnish), words are formed by long chains of suffixes. BPE often splits these in ways that break the semantic root. Byte-level models learn the continuous patterns of morphological addition.
2. **Typo-Robustness:** If a user types "tokenizashun", a BPE tokenizer might produce `['tokeniz', 'ash', 'un']`, which looks nothing like the standard `['tokenization']`. A byte-level model sees the high character-level similarity and generalizes better.
3. **Zero OOV (Out-Of-Vocabulary):** Because the 'vocabulary' is just the 256 bytes of UTF-8, the model can represent *every* possible string in existence. There is no such thing as an unknown word.

> **The Masterclass Trade-off:** The cost is **computational density**. A sentence of 10 words might be 15 tokens in BPE, but 100+ bytes in ByT5. This increases sequence length significantly, requiring more memory and compute (due to attention complexity). This is why BPE remains the 'Goldilocks' zone for efficiency... for now.


---
## ✅ Knowledge Check

### Q1: The OOV Paradox
Which algorithm handles the word "Hyper-loop" better if it was never in the training set: **Word-level** or **Subword (BPE)**?

<details><summary>👉 Answer</summary>
Subword (BPE). Word-level would flag it as [UNK] (Unknown). BPE would likely break it into ["Hyper", "-", "loop"], allowing the model to aggregate the meanings of those pieces.
</details>

### Q2: Mathematical Efficiency
Why does TF-IDF use the Logarithm for the Inverse Document Frequency? 

<details><summary>👉 Answer</summary>
Without the log, IDF values would grow linearly with the number of documents. For a corpus of 1 million docs where a word appears in only 1, the multiplier would be 1,000,000. This would drastically over-emphasize rare words. The log dampens this growth, making the weights manageable.
</details>

### Q3: Byte-Fallback (Senior Level)
How does Byte-Fallback help in preventing `[UNK]` for emojis like 🫠?

<details><summary>👉 Answer</summary>
If '🫠' is not in the vocab, the tokenizer converts it to its UTF-8 hex bytes (e.g., `\xf0\x9f\xaa\xa0`). Since all 256 bytes (0-255) are in the SentencePiece vocabulary, it can always represent the emoji as a sequence of 4 byte-tokens.
</details>

---

## 🔨 Practical Challenge

**Challenge:** Use the `augmentor` class you built to generate 3 augmented versions of the sentence: *"Artificial Intelligence is transforming the global economy at an unprecedented pace."*

--- 
**Next →** `16_02_word_embeddings_and_word2vec.ipynb` — NLP Masterclass 02: Word Embeddings & Word2Vec